# Experiment 13 — Standalone final Kaggle submission generator
 **Selected final submission:** logistic meta-stack of improved Experiment 08 LightGBM and public TabPFN-3.5, using probability and logit features. Historical cross-fitted OOF diagnostic **0.9613487673**. The earlier optimised blend reached **0.9604351410** OOF ROC AUC. **This notebook needs no earlier notebook state.** It consumes only CSV/NumPy prediction files, so it is fast and does **not** retrain LightGBM or TabPFN. This notebook *deliberately blocks writing a final Kaggle CSV* until the source/test-row-order, competition-rules and licence gates are reviewed. The same notebook can also write a conservative **25% LGB + 75% TabPFN** backup.

**Final Kaggle result:** 0.96177 private ROC AUC, rank 1,249 / 3,735. Public TabPFN predictions were externally sourced and their exact provenance/test-row alignment remains unverified.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Works inside your_project/notebooks/ and also when run next to the originally uploaded
# files, which may have parenthesised suffixes, e.g. train(3).csv.
CWD = Path.cwd().resolve()
SEARCH_ROOTS = list(dict.fromkeys([CWD, *CWD.parents, Path('/mnt/data')]))
FILE_CHOICES = {
 'train': ['data/raw/train.csv', 'train.csv', 'train(3).csv'],
 'test': ['data/raw/test.csv', 'test.csv', 'test(3).csv'],
 'sample': ['data/raw/sample_submission.csv', 'sample_submission.csv', 'sample_submission(3).csv'],
 'lgb_oof': ['submissions/exp08_flight_distance/08_best_candidate_oof.csv', '08_best_candidate_oof.csv', '08_best_candidate_oof(1).csv'],
 'lgb_test': ['submissions/exp08_flight_distance/08_best_candidate_test_components.csv', '08_best_candidate_test_components.csv', '08_best_candidate_test_components(1).csv'],
 'real_oof': ['submissions/realmlp_submission_v1/paired_oof_predictions.csv', 'paired_oof_predictions.csv', 'paired_oof_predictions(5).csv'],
 'real_test': ['submissions/realmlp_submission_v1/07_realmlp_submission.csv', '07_realmlp_submission.csv', '07_realmlp_submission(2).csv'],
 'tab_oof': ['submissions/public_tfm/oof_tabpfn35.npy', 'oof_tabpfn35.npy'],
 'tab_test': ['submissions/public_tfm/test_tabpfn35.npy', 'test_tabpfn35.npy'],
 'kumo_oof': ['submissions/public_tfm/oof_kumolarge.npy', 'oof_kumolarge.npy'],
 'kumo_test': ['submissions/public_tfm/test_kumolarge.npy', 'test_kumolarge.npy'],
 'exp09_test': ['submissions/exp09_optimised_weights/09_optimised_weights_submission.csv', '09_optimised_weights_submission.csv', '09_optimised_weights_submission(3).csv']
}

def locate(key, required=True):
    for base in SEARCH_ROOTS:
        for rel in FILE_CHOICES[key]:
            p = base / rel
            if p.is_file():
                return p
    if required:
        raise FileNotFoundError(f'Missing {key}. Looked for: {FILE_CHOICES[key]} in {SEARCH_ROOTS}')
    return None

def project_root():
    for base in SEARCH_ROOTS:
        if (base / 'data/raw/train.csv').is_file():
            return base
    # If running from this package's notebooks folder, save under the package root.
    return CWD.parent if CWD.name == 'notebooks' else CWD

OUTPUT_DIR = project_root() / 'submissions' / 'exp12_tfm_stack'

def check_prob(name, values, n):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    assert values.shape == (n,), f'{name} shape {values.shape} != ({n},)'
    assert np.isfinite(values).all(), f'{name} has NaN or inf'
    assert (values >= 0).all() and (values <= 1).all(), f'{name} must be probabilities'
    return values

def meta_features(lgb, tab):
    p = np.column_stack([lgb, tab]).astype(np.float64)
    pc = np.clip(p, 1e-6, 1-1e-6)
    return np.column_stack([p, np.log(pc/(1-pc))])

def fit_meta(M, y, train_mask, C=1.0):
    scaler = StandardScaler().fit(M[train_mask])
    model = LogisticRegression(C=C, solver='newton-cholesky', max_iter=100, tol=1e-7)
    model.fit(scaler.transform(M[train_mask]), y[train_mask])
    return scaler, model

def meta_predict(M, scaler, model):
    return model.predict_proba(scaler.transform(M))[:, 1]

def load_data(need_test=False, need_real=True, need_kumo=False):
    train = pd.read_csv(locate('train'), usecols=['id','satisfaction'])
    n = len(train)
    train_ids = train['id'].to_numpy()
    assert pd.Series(train_ids).is_unique, 'train IDs not unique'
    y = train['satisfaction'].astype(int).to_numpy()
    assert set(np.unique(y)) == {0, 1}, 'Unexpected target labels'
    lgb = pd.read_csv(locate('lgb_oof'), usecols=['id','satisfaction','fold','oof_LightGBM_improved'])
    assert np.array_equal(lgb['id'].to_numpy(), train_ids), 'LightGBM OOF row order / IDs mismatch'
    assert np.array_equal(lgb['satisfaction'].astype(int).to_numpy(), y), 'LightGBM target mismatch'
    fold = lgb['fold'].astype(int).to_numpy()
    assert set(fold) == {1,2,3,4,5}
    rebuilt = np.zeros(n, dtype=int)
    for fold_num, (_, val_idx) in enumerate(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(np.zeros(n),y),start=1):
        rebuilt[val_idx] = fold_num
    assert np.array_equal(rebuilt,fold), 'Fold assignment not seed-42 StratifiedKFold'
    o = dict(n=n, y=y, fold=fold, train_ids=train_ids, lgb_oof=check_prob('lgb_oof',lgb['oof_LightGBM_improved'],n))
    if need_real:
        real = pd.read_csv(locate('real_oof'), usecols=['id','satisfaction','fold','oof_RealMLP'])
        assert np.array_equal(real['id'].to_numpy(),train_ids), 'RealMLP OOF IDs mismatch'
        assert np.array_equal(real['satisfaction'].astype(int).to_numpy(),y), 'RealMLP target mismatch'
        assert np.array_equal(real['fold'].astype(int).to_numpy(),fold), 'RealMLP fold mismatch'
        o['real_oof'] = check_prob('real_oof',real['oof_RealMLP'],n)
        baseline_auc = roc_auc_score(y,0.67*o['lgb_oof']+0.33*o['real_oof'])
        assert abs(baseline_auc - 0.9604351410321921) < 5e-7, f'Exp09 baseline not reproduced: {baseline_auc}'
        o['exp09_oof'] = 0.67*o['lgb_oof'] + 0.33*o['real_oof']
    o['tab_oof'] = check_prob('TabPFN OOF',np.load(locate('tab_oof')),n)
    tab_auc = roc_auc_score(y,o['tab_oof'])
    assert abs(tab_auc-0.9611450967635963) < 5e-5, f'TabPFN OOF fingerprint mismatch: {tab_auc}'
    if need_kumo:
        o['kumo_oof'] = check_prob('Kumo OOF',np.load(locate('kumo_oof')),n)
    if need_test:
        test = pd.read_csv(locate('test'),usecols=['id'])
        template = pd.read_csv(locate('sample'))
        test_ids = test['id'].to_numpy()
        nt = len(test)
        assert np.array_equal(template['id'].to_numpy(),test_ids), 'sample_submission ID/order mismatch'
        lgb_test = pd.read_csv(locate('lgb_test'),usecols=['id','improved_LightGBM_test'])
        assert np.array_equal(lgb_test['id'].to_numpy(),test_ids), 'LightGBM test ID/order mismatch'
        o.update(test_ids=test_ids,n_test=nt,sample=template,
                 lgb_test=check_prob('lgb_test',lgb_test['improved_LightGBM_test'],nt),
                 tab_test=check_prob('TabPFN test',np.load(locate('tab_test')),nt))
        if need_real:
            rp = locate('real_test',required=False)
            if rp is not None:
                rd = pd.read_csv(rp,usecols=['id','satisfaction'])
                assert np.array_equal(rd['id'].to_numpy(),test_ids), 'RealMLP test ID/order mismatch'
                o['real_test'] = check_prob('real_test',rd['satisfaction'],nt)
                old = locate('exp09_test',required=False)
                if old is not None:
                    old_df=pd.read_csv(old,usecols=['id','satisfaction'])
                    assert np.array_equal(old_df['id'].to_numpy(),test_ids),'Old Exp09 ID/order mismatch'
                    maxerr=np.max(np.abs(old_df['satisfaction'].to_numpy() - (0.67*o['lgb_test']+0.33*o['real_test'])))
                    print(f'Exp09 test reconstruction max |difference|: {maxerr:.3g}')
                    assert maxerr < 1e-6
        if need_kumo:
            o['kumo_test'] = check_prob('Kumo test',np.load(locate('kumo_test')),nt)
    return o

## 1. Configuration and input checks
 The script accepts the canonical project structure or the supplied upload names. The `train.csv` IDs, owned-model OOF IDs/folds and LightGBM test IDs are asserted. TabPFN `.npy` arrays carry no IDs; **matching sizes and the OOF AUC fingerprint cannot establish test ordering**.

In [ ]:
d=load_data(need_test=True,need_real=False,need_kumo=False)
print('Inputs and owned model IDs checked.')
print('Improved LightGBM OOF:',format(roc_auc_score(d['y'],d['lgb_oof']),'.12f'))
print('TabPFN OOF:',format(roc_auc_score(d['y'],d['tab_oof']),'.12f'))
print('Train rows:',d['n'],'Test rows:',d['n_test'])

## 2. Reproduce the selected cross-fitted score (fast fingerprint)
 These fixed fold-wise `C` values were selected during Experiment 12, which is no longer available. This reproduces the historical diagnostic **given the original OOF arrays**; it cannot independently audit or repeat the original hyperparameter-selection procedure.

In [ ]:
M_train=meta_features(d['lgb_oof'],d['tab_oof'])
M_test=meta_features(d['lgb_test'],d['tab_test'])
SELECTED_C_PER_FOLD={1:1.0, 2:0.3, 3:0.1, 4:0.1, 5:0.3}
validation_pred=np.zeros(d['n'],dtype=np.float64)
for heldout,C_val in SELECTED_C_PER_FOLD.items():
    tr=d['fold']!=heldout; va=~tr
    sc,clf=fit_meta(M_train,d['y'],tr,C_val)
    validation_pred[va]=meta_predict(M_train[va],sc,clf)
    print(f'Fold {heldout}: {roc_auc_score(d["y"][va],validation_pred[va]):.9f} (C={C_val})')
validated_auc=roc_auc_score(d['y'],validation_pred)
print(f'Cross-fitted AUC: {validated_auc:.12f}')
assert abs(validated_auc-0.9613487672712965)<1e-6, 'Model fingerprint mismatch, do not submit'

## 3. Fit all-OOF final meta-model and compute final test predictions
 Full meta-CV chose `C=1.0`; that value and all input feature transformations are fixed here. The test prediction is **not** the OOF score. The selected submission achieved private Kaggle ROC AUC 0.96177; the submitted filenames included an unverified-test-order warning.

In [ ]:
FINAL_C=1.0
final_scale, final_model=fit_meta(M_train,d['y'],np.ones(d['n'],dtype=bool),FINAL_C)
pred_final=meta_predict(M_test,final_scale,final_model)
pred_backup=0.25*d['lgb_test']+0.75*d['tab_test']
assert pred_final.shape==(d['n_test'],)
assert np.isfinite(pred_final).all() and ((pred_final>=0)&(pred_final<=1)).all()
print('Final logistic prediction summary:')
print(pd.Series(pred_final).describe(percentiles=[.01,.25,.5,.75,.99]).to_string())
print('First five predictions:',np.round(pred_final[:5],8))

## 4. Public test-order and usage safeguards
 Before changing the flags to `True`, open the original public TabPFN Kaggle generation notebook or Data Card and confirm **all** of these: (a) OOF predictions are genuinely out-of-fold; (b) `test_tabpfn35.npy[i]` predicts `test.csv.iloc[i]`, *without shuffled rows*; (c) using the public predictions is allowed by the competition rules and TabPFN's applicable terms. Note that an AUC fingerprint validates train ordering very strongly but **cannot prove test ordering**. Keep the flags at `False` if any item remains uncertain.

In [ ]:
# ---- USER MUST REVIEW THESE FOUR FLAGS ----
PUBLIC_OOF_GENERATION_REVIEWED = False
PUBLIC_TEST_ORDER_CONFIRMED = False
COMPETITION_RULES_AND_MODEL_LICENSE_REVIEWED = False
ALLOW_SUBMISSION_WRITE = False

# Previews are allowed without writing any final Kaggle CSV:
submission_preview=d['sample'].copy()
submission_preview['satisfaction']=pred_final
print('Preview (not a saved submission):')
print(submission_preview.head().to_string(index=False))

requirements_ok=all([
    PUBLIC_OOF_GENERATION_REVIEWED,
    PUBLIC_TEST_ORDER_CONFIRMED,
    COMPETITION_RULES_AND_MODEL_LICENSE_REVIEWED,
    ALLOW_SUBMISSION_WRITE
])
if not requirements_ok:
    print('\nSUBMISSION WRITING BLOCKED: review provenance and explicitly set the flags True.')
else:
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    dest=OUTPUT_DIR/'12_submission_logistic_lgb_tabpfn.csv'
    submission_preview.to_csv(dest,index=False)
    backup=d['sample'].copy()
    backup['satisfaction']=pred_backup
    backup_dest=OUTPUT_DIR/'12_submission_grid_lgb25_tab75.csv'
    backup.to_csv(backup_dest,index=False)
    for written in (dest,backup_dest):
        verify=pd.read_csv(written)
        assert np.array_equal(verify['id'].to_numpy(),d['test_ids'])
        assert len(verify)==d['n_test']
    print('SUBMISSIONS WRITTEN:',dest,'and',backup_dest)

## Submission guidance
 Upload `12_submission_logistic_lgb_tabpfn.csv` to Kaggle **after** the source and rules checks pass. Keep `12_submission_grid_lgb25_tab75.csv` as a robust backup; its cross-fitted OOF was **0.96131015**, only about 0.000039 below the logistic model. The selected logistic-stack submission scored 0.96177 on the private leaderboard. Do not infer performance for newly generated CSVs without verifying data provenance and test alignment.